In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from phase import PhaseSolver, PhaseConfig

def correct(frame, mask):
    ys, xs = np.where(mask)
    corr = frame.astype(float).copy()
    for y, x in zip(ys, xs):
        nb  = frame[max(0,y-1):y+2, max(0,x-1):x+2]
        nbm = mask [max(0,y-1):y+2, max(0,x-1):x+2]
        good = nb[~nbm]
        if good.size: corr[y,x] = np.median(good)
    return corr

N = 30

In [ ]:
# Read dark frames
data_dark = np.load("../../data/dark_400_20260913_171239.npz")
stack_dark = data_dark['stack'][0]

In [ ]:
# Find and correct for hot pixels
frames = stack_dark           # (N,H,W), same gain+exposure as your data

m = frames.mean(0)                 # per-pixel mean  -> high dark current
s = frames.std(0)                  # per-pixel std   -> RTS/flicker

def rob(x, k=5):                   # robust threshold, unbiased by outliers
    med = np.median(x)
    mad = np.median(np.abs(x-med))*1.4826
    return np.abs(x-med) >= k*mad

mask = rob(m) | rob(s)             # (H,W) bool, True = hot
print(100*mask.mean(), "% hot")

stack_dark_corr = np.stack([correct(stack_dark[n], mask) for n in range(len(stack_dark))])
stack_dark_ref = stack_dark_corr.mean(axis=0)

In [ ]:
# Read ref bright frames
data_ref_bright = np.load("../../data/piezo_bright_400_20260913_171406.npz")
stack_ref_bright = data_ref_bright['stack'][0][:N]

In [ ]:
# Correct for hot pixels and subtract dark
stack_ref_bright_corr = np.stack([correct(stack_ref_bright[n], mask) for n in range(len(stack_ref_bright))])
stack_ref_bright_corr = np.maximum(stack_ref_bright_corr - stack_dark_ref, 0)

plt.figure(figsize=(5, 5))
plt.plot(stack_ref_bright_corr[0,1000])
plt.plot(stack_ref_bright_corr[3,1000])

In [ ]:
# Compute noise
sigma_ref_n = np.sqrt(stack_ref_bright_corr*2.6596) / 2.6596    # (N,P), gain=100
sigma_ref_pixel = np.sqrt((sigma_ref_n**2).mean(0))          # (P,)

rms_ref_bright = np.sqrt((sigma_ref_pixel**2).mean())
noise_floor_ref_bright = np.sqrt((sigma_ref_pixel**2).mean()) / stack_ref_bright_corr.std()

print(f"RMS noise ref: {rms_ref_bright}")
print(f"Noise floor ref: {noise_floor_ref_bright}")

In [ ]:
num_range = np.arange(start=5, stop=31)
iters = 30
tol = 1e-4
refine_iters = 10
refine_tol = 1e-4

cfg_d = PhaseConfig.from_yaml("phase_config.yaml")
cfg_d.use_alpha = True
cfg_d.gain_mode = "joint"
cfg_d.method_kwargs.update(iters=iters, tol=tol, degree=2,
                            refine_iters=refine_iters, refine_tol=refine_tol)

results_ref = {}
for num in num_range:
    solver_d = PhaseSolver(cfg_d)
    solver_d.fit(stack_ref_bright_corr[:num])
    results_ref[num] = solver_d
    mp = solver_d.method_param_
    print(f"num={num} converged={mp.refine_converged}  reconstruction_error={solver_d.reconstruction_error_:.4f}  "
            f"kappa_fit={mp.kappa_fit:.3g}  refine_iters_run={mp.refine_iters_run}  "
            f"rms_frac_history={[f'{x:.5f}' for x in mp.rms_frac_history]}")

In [ ]:
num_range = np.arange(start=5, stop=31)
iters = 30
tol = 1e-4
refine_iters = 10
refine_tol = 1e-4

cfg_d = PhaseConfig.from_yaml("phase_config.yaml")
cfg_d.use_alpha = False
cfg_d.gain_mode = None#"joint"
cfg_d.method_kwargs.update(iters=iters, tol=tol, degree=0,
                            refine_iters=refine_iters, refine_tol=refine_tol)

results_ref0_alpha_g = {}
for num in num_range:
    solver_d = PhaseSolver(cfg_d)
    solver_d.fit(stack_ref_bright_corr[:num])
    results_ref0_alpha_g[num] = solver_d
    mp = solver_d.method_param_
    print(f"num={num} converged={mp.refine_converged}  reconstruction_error={solver_d.reconstruction_error_:.4f}  "
            f"kappa_fit={mp.kappa_fit:.3g}  refine_iters_run={mp.refine_iters_run}  "
            f"rms_frac_history={[f'{x:.5f}' for x in mp.rms_frac_history]}")

In [ ]:
plt.plot(num_range, [results_ref[num].reconstruction_error_*np.sqrt(num/(num-3)) for num in num_range],'.-', color="tab:blue")
# plt.plot(num_range, [results_ref0[num].reconstruction_error_*np.sqrt(1-3/num) for num in num_range],'.-', color="tab:orange")
# plt.plot(num_range, [results_ref0_alpha[num].reconstruction_error_*np.sqrt(1-3/num) for num in num_range],'.-', color="tab:green")
# plt.plot(num_range, [results_ref0_alpha_g[num].reconstruction_error_*np.sqrt(1-3/num) for num in num_range],'.-', color="tab:red")
plt.plot(num_range, rms_ref_bright_arr)
plt.hlines(rms_ref_bright, 5, 30, color='tab:blue', linestyles='--')
#plt.text(6.5,8.7, "Noise floor")
plt.xlim([5, 30])
plt.legend()
plt.xlabel('Number of piezo steps (frames)')
plt.ylabel('Reconstruction error, RMS')

In [ ]:
plt.plot(results_ref[30].method_param_, '.-')

In [ ]:
a = results_ref[20].a_
b = results_ref[20].b_
phi = results_ref[20].phi_
phi_error = results_ref[20].phi_error_
contrast = b/a

plt.imshow(contrast)

In [ ]:
plt.imshow(phi_error*180/np.pi)
plt.clim([0, 1])

print(np.sqrt(np.mean(phi_error**2))*180/np.pi)

In [ ]:
from phase.carrier import remove_carrier

phi_arr = np.asarray([remove_carrier(results_ref[i+5].phi_ * (results_ref[i+5].phi_.mean()/np.abs(results_ref[i+5].phi_.mean())),results_ref[i+5].b_).phi  for i in range(26)])
phi_diff = np.diff(phi_arr, axis=0)
phi_diff_error = np.mean(phi_diff**2, axis=(1,2))

plt.plot(phi_diff_error)
# plt.ylim([0, 0.1])

In [ ]:
plt.plot(np.arange(5, 30), phi_diff_error*180/np.pi)

In [ ]:
phi_arr[0].mean()/np.abs(phi_arr[0].mean())

In [ ]:
from phase.carrier import remove_carrier

res = remove_carrier(phi, b)

In [ ]:
plt.imshow(res.phi)

In [ ]:
plt.plot(res.phi[0])